In [5]:
"""
Train a Keras autoencoder for STM32Cube.AI and future AKD/MetaTF portability.

Recommended local setup for STM32Cube.AI export:
    python -m venv .venv
    .venv\\Scripts\\activate        # Windows
    source .venv/bin/activate        # Linux/macOS
    pip install -r requirements.txt

Typical execution:
    python 01_training/train_autoencoder_keras.py \
        --csv data/tiltmonitor_incidence_series_24h.csv \
        --out artifacts \
        --epochs 200 \
        --threshold-mode normal-percentile \
        --threshold-percentile 99

Outputs:
    artifacts/stm32cube_ai/autoencoder.keras
    artifacts/stm32cube_ai/autoencoder.h5              (if backend supports it)
    artifacts/stm32cube_ai/autoencoder_float32.tflite  (if TensorFlow is available)
    artifacts/stm32cube_ai/autoencoder_int8.tflite     (if TensorFlow is available)
    artifacts/common/normalization_params.json
    artifacts/common/ae_postprocess_params.h
    artifacts/common/training_metadata.json
    artifacts/validation/window_features.csv
    artifacts/validation/validation_vectors.csv
"""

from __future__ import annotations

import argparse
import json
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.model_selection import train_test_split

from pathlib import Path
import sys

# Añadimos la subcarpeta 01_training al camino de búsqueda
training_dir = Path.cwd() / "01_training" if (Path.cwd() / "01_training").exists() else Path.cwd()
if str(training_dir) not in sys.path:
    sys.path.insert(0, str(training_dir))

# Importamos las funciones desde el nuevo feature_engineering.py
from feature_engineering import (
    FEATURE_NAMES,
    WindowConfig,
    build_windows_from_csv,
    save_feature_csv,
)

print("¡Importado con éxito desde feature_engineering.py!")


def import_keras():
    """Import Keras.

    For final STM32Cube.AI work, TensorFlow backend is recommended.
    In environments without TensorFlow, Keras 3 can also run with torch/jax backend,
    but TFLite export will not be available.
    """
    try:
        import keras  # type: ignore
    except Exception as exc:
        raise RuntimeError(
            "Could not import keras. Install TensorFlow/Keras first, or set a Keras 3 backend. "
            "Recommended for STM32Cube.AI: pip install tensorflow keras"
        ) from exc
    return keras


def build_autoencoder(keras, input_dim: int = 14):
    inp = keras.Input(shape=(input_dim,), name="ae_input")
    x = keras.layers.Dense(8, activation="relu", name="encoder_dense_8")(inp)
    x = keras.layers.Dense(4, activation="relu", name="bottleneck_dense_4")(x)
    x = keras.layers.Dense(8, activation="relu", name="decoder_dense_8")(x)
    out = keras.layers.Dense(input_dim, activation="linear", name="ae_reconstruction")(x)
    model = keras.Model(inp, out, name="tiltmeter_autoencoder_14_8_4_8_14")
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3), loss="mse")
    return model


def standardize_train_apply(X_train_normal: np.ndarray, X_all: np.ndarray, eps: float = 1e-6):
    mean = X_train_normal.mean(axis=0).astype(np.float32)
    std = X_train_normal.std(axis=0, ddof=0).astype(np.float32)
    std = np.maximum(std, eps).astype(np.float32)
    X_all_norm = ((X_all - mean) / std).astype(np.float32)
    return X_all_norm, mean, std


def reconstruction_mse(model, X_norm: np.ndarray, batch_size: int = 256) -> np.ndarray:
    X_hat = model.predict(X_norm, batch_size=batch_size, verbose=0)
    err = np.mean((X_norm - X_hat) ** 2, axis=1)
    return err.astype(np.float32)


def choose_threshold(errors_val: np.ndarray, y_val: np.ndarray, mode: str, percentile: float):
    normal_errors = errors_val[y_val == 0]
    if normal_errors.size == 0:
        raise ValueError("No normal validation windows available to define threshold")

    if mode == "normal-percentile":
        threshold = float(np.percentile(normal_errors, percentile))
        return threshold, {"mode": mode, "percentile": percentile}

    if mode == "best-f1":
        candidates = np.unique(np.percentile(errors_val, np.linspace(1, 99.9, 500)))
        best = {"threshold": float(candidates[0]), "f1": -1.0}
        for th in candidates:
            pred = (errors_val > th).astype(np.int64)
            _, _, f1, _ = precision_recall_fscore_support(y_val, pred, average="binary", zero_division=0)
            if f1 > best["f1"]:
                best = {"threshold": float(th), "f1": float(f1)}
        return float(best["threshold"]), {"mode": mode, "best_f1": best["f1"]}

    raise ValueError(f"Unknown threshold mode: {mode}")


def write_postprocess_header(output_path: Path, mean: np.ndarray, std: np.ndarray, threshold: float):
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("w", encoding="utf-8") as f:
        f.write("#ifndef AE_POSTPROCESS_PARAMS_H\n")
        f.write("#define AE_POSTPROCESS_PARAMS_H\n\n")
        f.write("#include <stdint.h>\n\n")
        f.write("#define AE_INPUT_SIZE 14u\n")
        f.write(f"#define AE_RECONSTRUCTION_THRESHOLD_F32 ({threshold:.9g}f)\n\n")
        f.write("static const float ae_input_mean[AE_INPUT_SIZE] = {\n")
        f.write(",\n".join([f"    {v:.9g}f" for v in mean]))
        f.write("\n};\n\n")
        f.write("static const float ae_input_std[AE_INPUT_SIZE] = {\n")
        f.write(",\n".join([f"    {v:.9g}f" for v in std]))
        f.write("\n};\n\n")
        f.write("#endif /* AE_POSTPROCESS_PARAMS_H */\n")


def try_export_tflite(model, X_train_norm: np.ndarray, out_dir: Path):
    """Export float32 and INT8 TFLite if TensorFlow is available."""
    try:
        import tensorflow as tf  # type: ignore
    except Exception:
        return {
            "tflite_float32": False,
            "tflite_int8": False,
            "reason": "TensorFlow not available in this Python environment",
        }

    results = {}

    try:
        converter = tf.lite.TFLiteConverter.from_keras_model(model)
        tflite_model = converter.convert()
        (out_dir / "autoencoder_float32.tflite").write_bytes(tflite_model)
        results["tflite_float32"] = True
    except Exception as exc:
        results["tflite_float32"] = False
        results["tflite_float32_error"] = str(exc)

    try:
        def representative_dataset():
            n = min(256, len(X_train_norm))
            for i in range(n):
                yield [X_train_norm[i:i+1].astype(np.float32)]

        converter = tf.lite.TFLiteConverter.from_keras_model(model)
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.representative_dataset = representative_dataset
        converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
        converter.inference_input_type = tf.int8
        converter.inference_output_type = tf.int8
        tflite_int8 = converter.convert()
        (out_dir / "autoencoder_int8.tflite").write_bytes(tflite_int8)
        results["tflite_int8"] = True
    except Exception as exc:
        results["tflite_int8"] = False
        results["tflite_int8_error"] = str(exc)

    return results


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--csv", required=True, help="Path to historical CSV")
    parser.add_argument("--out", default="artifacts", help="Output artifacts directory")
    parser.add_argument("--window-size", type=int, default=4)
    parser.add_argument("--stride", type=int, default=1)
    parser.add_argument("--epochs", type=int, default=200)
    parser.add_argument("--batch-size", type=int, default=64)
    parser.add_argument("--threshold-mode", choices=["normal-percentile", "best-f1"], default="normal-percentile")
    parser.add_argument("--threshold-percentile", type=float, default=99.0)
    parser.add_argument("--seed", type=int, default=7)
    parser.add_argument("--verbose", type=int, default=0)
    args = parser.parse_args()

    np.random.seed(args.seed)

    out = Path(args.out)
    stm_dir = out / "stm32cube_ai"
    common_dir = out / "common"
    val_dir = out / "validation"
    for d in [stm_dir, common_dir, val_dir]:
        d.mkdir(parents=True, exist_ok=True)

    X, y, meta = build_windows_from_csv(args.csv, WindowConfig(args.window_size, args.stride))
    save_feature_csv(X, y, meta, val_dir / "window_features.csv")

    normal_idx = np.where(y == 0)[0]
    if normal_idx.size < 10:
        raise ValueError("Not enough normal windows for autoencoder training")

    idx_train_norm, idx_val_norm = train_test_split(
        normal_idx,
        test_size=0.25,
        random_state=args.seed,
        shuffle=True,
    )

    # Validation uses all non-training-normal windows: held-out normal + all anomalous windows.
    all_idx = np.arange(len(X))
    val_mask = np.ones(len(X), dtype=bool)
    val_mask[idx_train_norm] = False
    idx_val_all = all_idx[val_mask]

    X_train_normal = X[idx_train_norm]
    X_norm_all, mean, std = standardize_train_apply(X_train_normal, X)
    X_train_norm = X_norm_all[idx_train_norm]
    X_val_norm = X_norm_all[idx_val_all]
    y_val = y[idx_val_all]

    keras = import_keras()
    print(f"Using Keras {keras.__version__} with backend: {keras.backend.backend()}")

    model = build_autoencoder(keras, input_dim=14)
    callbacks = []
    try:
        callbacks.append(keras.callbacks.EarlyStopping(monitor="val_loss", patience=20, restore_best_weights=True))
    except Exception:
        pass

    history = model.fit(
        X_train_norm,
        X_train_norm,
        validation_split=0.2,
        epochs=args.epochs,
        batch_size=args.batch_size,
        shuffle=True,
        verbose=args.verbose,
        callbacks=callbacks,
    )

    errors_val = reconstruction_mse(model, X_val_norm)
    threshold, threshold_info = choose_threshold(
        errors_val,
        y_val,
        args.threshold_mode,
        args.threshold_percentile,
    )
    y_pred = (errors_val > threshold).astype(np.int64)
    acc = accuracy_score(y_val, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(y_val, y_pred, average="binary", zero_division=0)

    # Save Keras model. .keras is the primary STM32Cube.AI-friendly artifact.
    keras_path = stm_dir / "autoencoder.keras"
    model.save(keras_path)

    # Try legacy .h5 as a secondary artifact if supported by installed backend.
    h5_saved = False
    try:
        model.save(stm_dir / "autoencoder.h5")
        h5_saved = True
    except Exception as exc:
        print(f"Could not save .h5 artifact: {exc}")

    tflite_results = try_export_tflite(model, X_train_norm, stm_dir)

    norm_json = {
        "feature_names": FEATURE_NAMES,
        "mean": mean.astype(float).tolist(),
        "std": std.astype(float).tolist(),
        "input_note": "The STM32Cube.AI model expects normalized x[14]. Normalize in firmware before network inference.",
    }
    (common_dir / "normalization_params.json").write_text(json.dumps(norm_json, indent=2), encoding="utf-8")

    threshold_json = {
        "threshold_mse_normalized_space": threshold,
        "decision_rule": "SEND if reconstruction_mse > threshold",
        "threshold_info": threshold_info,
    }
    (common_dir / "ae_threshold.json").write_text(json.dumps(threshold_json, indent=2), encoding="utf-8")
    write_postprocess_header(common_dir / "ae_postprocess_params.h", mean, std, threshold)

    # Save a small set of validation vectors for C/integration tests.
    rng = np.random.default_rng(args.seed)
    sample_n = min(32, len(idx_val_all))
    sample_idx = rng.choice(idx_val_all, size=sample_n, replace=False)
    val_vectors = pd.DataFrame(X[sample_idx], columns=FEATURE_NAMES)
    val_vectors.insert(0, "label", y[sample_idx])
    val_vectors.to_csv(val_dir / "validation_vectors.csv", index=False)

    metadata = {
        "model": "autoencoder_14_8_4_8_14",
        "train_script": "train_autoencoder_keras.py",
        "keras_version": str(keras.__version__),
        "keras_backend": str(keras.backend.backend()),
        "csv": str(args.csv),
        "window_size": args.window_size,
        "stride": args.stride,
        "n_windows": int(len(X)),
        "n_train_normal_windows": int(len(idx_train_norm)),
        "n_validation_windows": int(len(idx_val_all)),
        "n_validation_anomalous_windows": int((y_val == 1).sum()),
        "threshold": float(threshold),
        "metrics_validation": {
            "accuracy": float(acc),
            "precision_send": float(precision),
            "recall_send": float(recall),
            "f1_send": float(f1),
        },
        "saved_artifacts": {
            "keras": str(keras_path),
            "h5": h5_saved,
            "tflite_results": tflite_results,
        },
        "feature_names": FEATURE_NAMES,
    }
    (common_dir / "training_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")

    print("\nTraining completed.")
    print(json.dumps(metadata, indent=2))

import sys

# 1. Ajustamos la ruta relativa hacia atrás (..) para encontrar la carpeta data/
sys.argv = [
    "train_autoencoder_keras.py",
    "--csv", "../data/tiltmonitor_incidence_series_24h.csv",
    "--out", "../artifacts",
    "--epochs", "200",
    "--threshold-mode", "normal-percentile",
    "--threshold-percentile", "99"
]

# 2. Iniciamos el entrenamiento
if __name__ == "__main__":
    main()

¡Importado con éxito desde feature_engineering.py!
Using Keras 3.15.1 with backend: tensorflow


INFO:tensorflow:Assets written to: C:\Users\maryp\AppData\Local\Temp\tmptlvhsjc0\assets


INFO:tensorflow:Assets written to: C:\Users\maryp\AppData\Local\Temp\tmptlvhsjc0\assets


Saved artifact at 'C:\Users\maryp\AppData\Local\Temp\tmptlvhsjc0'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 14), dtype=tf.float32, name='ae_input')
Output Type:
  TensorSpec(shape=(None, 14), dtype=tf.float32, name=None)
Captures:
  1489273875408: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1489273876560: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1489273875216: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1489273875024: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1489273875984: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1489273874832: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1488790642256: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1489273876368: TensorSpec(shape=(), dtype=tf.resource, name=None)
INFO:tensorflow:Assets written to: C:\Users\maryp\AppData\Local\Temp\tmp2xkfccgt\assets


INFO:tensorflow:Assets written to: C:\Users\maryp\AppData\Local\Temp\tmp2xkfccgt\assets


Saved artifact at 'C:\Users\maryp\AppData\Local\Temp\tmp2xkfccgt'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 14), dtype=tf.float32, name='ae_input')
Output Type:
  TensorSpec(shape=(None, 14), dtype=tf.float32, name=None)
Captures:
  1489273875408: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1489273876560: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1489273875216: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1489273875024: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1489273875984: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1489273874832: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1488790642256: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1489273876368: TensorSpec(shape=(), dtype=tf.resource, name=None)

Training completed.
{
  "model": "autoencoder_14_8_4_8_14",
  "train_script": "train_autoencoder_keras.py",
  "keras_version": "3.15.1",
  "keras_backend"

C:\Users\maryp\anaconda3\Lib\site-packages\tensorflow\lite\python\convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
